# Crawl Luật Đường Bộ từ PDF

Notebook này sẽ tải xuống và trích xuất nội dung từ file PDF Luật Đường Bộ số 35/2024/QH15 và lưu vào file JSON.

## 1. Cài đặt và Import thư viện

In [9]:
# Cài đặt thư viện
import sys
!{sys.executable} -m pip install -q PyPDF2 requests pdfplumber

'c:\Program' is not recognized as an internal or external command,
operable program or batch file.


In [10]:
# Import các thư viện cần thiết
import requests
import json
import re
from pathlib import Path
import pdfplumber
import sys
import io

## 2. Tải xuống file PDF

In [11]:
# URL và tên file PDF
pdf_url = "https://datafiles.chinhphu.vn/cpp/files/vbpq/2024/9/35-2024-qh15.pdf"
pdf_filename = "35-2024-qh15.pdf"

# Kiểm tra file đã tồn tại chưa
pdf_path = Path(pdf_filename)

if pdf_path.exists():
    print(f"File PDF đã tồn tại: {pdf_filename}")
    print(f"Kích thước file: {pdf_path.stat().st_size / 1024:.2f} KB")
    # Đọc file có sẵn
    pdf_content = pdf_path.read_bytes()
else:
    print("File PDF chưa có, đang tải xuống...")
    response = requests.get(pdf_url)
    
    if response.status_code == 200:
        pdf_content = response.content
        # Lưu file để lần sau không cần tải lại
        pdf_path.write_bytes(pdf_content)
        print(f"Tải xuống và lưu thành công: {pdf_filename}")
        print(f"Kích thước file: {len(pdf_content) / 1024:.2f} KB")
    else:
        print(f"Lỗi tải xuống: {response.status_code}")
        raise Exception(f"Không thể tải file PDF")

print("✓ Sẵn sàng trích xuất dữ liệu")

File PDF đã tồn tại: 35-2024-qh15.pdf
Kích thước file: 725.71 KB
✓ Sẵn sàng trích xuất dữ liệu


## 3. Trích xuất văn bản từ PDF

In [12]:
# Trích xuất văn bản từ PDF trong memory
print("Đang trích xuất văn bản từ PDF...")

all_text = ""
pages_data = []

try:
    # Đọc PDF từ bytes trong memory
    pdf_file = io.BytesIO(pdf_content)
    
    with pdfplumber.open(pdf_file) as pdf:
        print(f"Tổng số trang: {len(pdf.pages)}")
        
        for i, page in enumerate(pdf.pages):
            page_text = page.extract_text()
            if page_text:
                pages_data.append({
                    "page_number": i + 1,
                    "content": page_text
                })
                all_text += page_text + "\n\n"
        
        print(f"Trích xuất thành công {len(pages_data)} trang")
        print(f"Độ dài văn bản: {len(all_text)} ký tự")
except Exception as e:
    print(f"Lỗi khi trích xuất PDF: {e}")
    raise

Đang trích xuất văn bản từ PDF...
Tổng số trang: 69
Trích xuất thành công 69 trang
Độ dài văn bản: 149931 ký tự


## 4. Phân tích và cấu trúc dữ liệu

In [13]:
# Tạo cấu trúc dữ liệu
law_data = {
    "title": "LUẬT TRẬT TỰ, AN TOÀN GIAO THÔNG ĐƯỜNG BỘ",
    "law_number": "35/2024/QH15",
    "year": 2024,
    "source_url": pdf_url,
    "total_pages": len(pages_data),
    "extraction_date": "2026-02-03",
    "full_text": all_text,
    "pages": pages_data
}

# Tìm các chương và điều
chapters = []
current_chapter = None

# Pattern để tìm chương (có thể điều chỉnh tùy theo format của PDF)
chapter_pattern = re.compile(r'Chương\s+([IVXLCDM]+|[0-9]+)\s*[\n:]?\s*(.+?)(?=\n|$)', re.IGNORECASE)
article_pattern = re.compile(r'Điều\s+(\d+)\s*\.?\s*(.+?)(?=\n|$)', re.IGNORECASE)

lines = all_text.split('\n')
current_chapter = None
current_article = None

for line in lines:
    # Kiểm tra chương
    chapter_match = chapter_pattern.search(line)
    if chapter_match:
        if current_chapter:
            chapters.append(current_chapter)
        current_chapter = {
            "chapter_number": chapter_match.group(1),
            "chapter_title": chapter_match.group(2).strip(),
            "articles": []
        }
        continue
    
    # Kiểm tra điều
    article_match = article_pattern.search(line)
    if article_match and current_chapter:
        current_article = {
            "article_number": article_match.group(1),
            "article_title": article_match.group(2).strip(),
            "content": ""
        }
        current_chapter["articles"].append(current_article)
        continue
    
    # Thêm nội dung vào điều hiện tại
    if current_article and line.strip():
        current_article["content"] += line + " "

# Thêm chương cuối cùng
if current_chapter:
    chapters.append(current_chapter)

law_data["chapters"] = chapters
print(f"Đã phân tích {len(chapters)} chương")

Đã phân tích 5 chương


## 5. Lưu vào file JSON

In [14]:
# Chuẩn bị dữ liệu JSON
json_filename = "luat_duong_bo_35-2024.json"
print(f"Đang chuẩn bị dữ liệu để lưu vào {json_filename}...")

# Chuyển sang JSON string
json_string = json.dumps(law_data, ensure_ascii=False, indent=2)
print(f"✓ Đã tạo JSON string: {len(json_string) / 1024:.2f} KB")
print("\nChạy cell tiếp theo để lưu file (dùng magic command, không bị Windows chặn)")

Đang chuẩn bị dữ liệu để lưu vào luat_duong_bo_35-2024.json...
✓ Đã tạo JSON string: 448.67 KB

Chạy cell tiếp theo để lưu file (dùng magic command, không bị Windows chặn)


In [15]:
# Kiểm tra thư mục và ghi file
import os
from pathlib import Path

print("=== KIỂM TRA THƯ MỤC ===")
current_dir = os.getcwd()
print(f"Thư mục hiện tại: {current_dir}")
print(f"Thư mục tồn tại: {os.path.exists(current_dir)}")
print(f"Có quyền ghi: {os.access(current_dir, os.W_OK)}")

# Tạo đường dẫn tuyệt đối
json_fullpath = os.path.join(current_dir, json_filename)
print(f"\nĐường dẫn đầy đủ: {json_fullpath}")

# Thử ghi file bằng builtins.open (bypass IPython)
print("\n=== GHI FILE ===")
import builtins
try:
    with builtins.open(json_fullpath, 'w', encoding='utf-8') as f:
        f.write(json_string)
    print(f"✓ Đã lưu thành công vào {json_filename}")
    
    # Kiểm tra file
    if Path(json_fullpath).exists():
        file_size = Path(json_fullpath).stat().st_size
        print(f"✓ Xác nhận file tồn tại: {file_size / 1024:.2f} KB")
        print(f"✓ Đường dẫn: {json_fullpath}")
    else:
        print("✗ File chưa được tạo")
        
except Exception as e:
    print(f"✗ Lỗi: {type(e).__name__}: {e}")
    print("\n=== DANH SÁCH FILE HIỆN TẠI ===")
    for item in os.listdir(current_dir):
        print(f"  - {item}")

=== KIỂM TRA THƯ MỤC ===
Thư mục hiện tại: c:\Users\ASUS-PRO\Documents\BaiHoc\Ki2Nam4\crawl_luat_duong_bo
Thư mục tồn tại: True
Có quyền ghi: True

Đường dẫn đầy đủ: c:\Users\ASUS-PRO\Documents\BaiHoc\Ki2Nam4\crawl_luat_duong_bo\luat_duong_bo_35-2024.json

=== GHI FILE ===
✓ Đã lưu thành công vào luat_duong_bo_35-2024.json
✓ Xác nhận file tồn tại: 597.06 KB
✓ Đường dẫn: c:\Users\ASUS-PRO\Documents\BaiHoc\Ki2Nam4\crawl_luat_duong_bo\luat_duong_bo_35-2024.json


## 6. Kiểm tra kết quả

In [16]:
# Hiển thị thông tin từ dữ liệu đã xử lý
print("\n=== THÔNG TIN LUẬT ===")
print(f"Tiêu đề: {law_data['title']}")
print(f"Số hiệu: {law_data['law_number']}")
print(f"Năm: {law_data['year']}")
print(f"Tổng số trang: {law_data['total_pages']}")
print(f"Số chương: {len(law_data.get('chapters', []))}")

# Hiển thị một số chương đầu tiên
if law_data.get('chapters'):
    print("\n=== MỘT SỐ CHƯƠNG ===")
    for chapter in law_data['chapters'][:3]:
        print(f"\nChương {chapter['chapter_number']}: {chapter['chapter_title']}")
        print(f"  - Số điều: {len(chapter.get('articles', []))}")
        if chapter.get('articles'):
            for article in chapter['articles'][:2]:
                print(f"    Điều {article['article_number']}: {article['article_title']}")

print("\n✅ Hoàn thành! Dữ liệu đã được xử lý.")


=== THÔNG TIN LUẬT ===
Tiêu đề: LUẬT TRẬT TỰ, AN TOÀN GIAO THÔNG ĐƯỜNG BỘ
Số hiệu: 35/2024/QH15
Năm: 2024
Tổng số trang: 69
Số chương: 5

=== MỘT SỐ CHƯƠNG ===

Chương I: I
  - Số điều: 50
    Điều 8: Phân loại đường bộ theo cấp quản lý
    Điều 9: Phân loại đường bộ theo chức năng phục vụ

Chương II: I
  - Số điều: 10
    Điều 44: Quy định chung đối với đường bộ cao tốc
    Điều 13: của Luật này;

Chương II: của Luật này và các quy định sau đây:
  - Số điều: 8
    Điều 50: Phí sử dụng đường cao tốc
    Điều 45: và khoản 2 Điều 47 của Luật này.

✅ Hoàn thành! Dữ liệu đã được xử lý.
